# MWAMNet — Stage 4: computational efficiency

In [ ]:
#@title 1 · Setup
!pip -q install kagglehub psutil

import os, json, time, gc, math
import numpy as np
import pandas as pd
import tensorflow as tf
import psutil

for _g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception as e:
        print("memory-growth setting skipped:", e)

S       = 224
WARMUP  = 50
TIMED   = 200
OUT     = "/content/MWAMNet_revision"
os.makedirs(os.path.join(OUT, "results"), exist_ok=True)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MIRROR = "/content/drive/MyDrive/MWAMNet_revision/results"
    os.makedirs(MIRROR, exist_ok=True)
except Exception as e:
    MIRROR = None
    print("Drive unavailable (%s) - local only." % e)

print("TensorFlow:", tf.__version__)
print("GPU       :", tf.config.list_physical_devices("GPU") or "NONE  <-- switch to T4 GPU")
print("CPU cores :", psutil.cpu_count(logical=True))

Mounted at /content/drive
TensorFlow: 2.20.0
GPU       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
CPU cores : 2


In [ ]:
#@title 2 · Deployable model definitions
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (Dense, BatchNormalization, Dropout,
                                     Concatenate, Lambda)
from tensorflow.keras.regularizers import l2 as L2reg
from tensorflow.keras.applications import (densenet, mobilenet_v3, resnet50,
                                           vgg16, inception_v3)
from tensorflow.keras.applications import (DenseNet201, MobileNetV3Large,
                                           ResNet50, VGG16, InceptionV3)

BACKBONES = {
    "DenseNet201":      (DenseNet201,      densenet.preprocess_input),
    "MobileNetV3Large": (MobileNetV3Large, mobilenet_v3.preprocess_input),
    "ResNet50":         (ResNet50,         resnet50.preprocess_input),
    "VGG16":            (VGG16,            vgg16.preprocess_input),
    "InceptionV3":      (InceptionV3,      inception_v3.preprocess_input),
}

MODELS = {
    "MWAMNet":          ["DenseNet201", "MobileNetV3Large"],
    "DenseNet201":      ["DenseNet201"],
    "MobileNetV3Large": ["MobileNetV3Large"],
    "ResNet50":         ["ResNet50"],
    "VGG16":            ["VGG16"],
    "InceptionV3":      ["InceptionV3"],
}

def build_deployable(bbs, l2v=0.01, drop=0.5):
    inp, feats = Input(shape=(S, S, 3), name="image"), []
    for i, name in enumerate(bbs):
        Base, prep = BACKBONES[name]
        net = Base(weights="imagenet", include_top=False,
                   input_shape=(S, S, 3), pooling="avg")
        net.trainable = False
        x = Lambda(prep, name="preprocess_%s" % name)(inp)
        f = net(x)
        f = Dropout(drop)(BatchNormalization()(f))
        feats.append(f)
    x = feats[0] if len(feats) == 1 else Concatenate()(feats)
    for u in (1024, 512):
        x = Dense(u, activation="relu", kernel_regularizer=L2reg(l2v))(x)
        x = Dropout(drop)(BatchNormalization()(x))
    return Model(inp, Dense(1, activation="sigmoid")(x), name=bbs[0] if len(bbs) == 1 else "MWAMNet")

_m = build_deployable(MODELS["MWAMNet"])
print("MWAMNet deployable pipeline built.")
print("  total parameters    : {:,}".format(_m.count_params()))
print("  trainable parameters: {:,}".format(
      int(sum(np.prod(w.shape) for w in _m.trainable_weights))))
del _m; tf.keras.backend.clear_session(); gc.collect()

74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MWAMNet deployable pipeline built.
  total parameters    : 24,811,457
  trainable parameters: 3,484,289


0

In [ ]:
#@title 3 · Parameters, FLOPs and serialised size
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

def count_flops(model):
    try:
        from tensorflow.python.profiler import model_analyzer, option_builder
        f = tf.function(lambda x: model(x, training=False))
        cf = f.get_concrete_function(tf.TensorSpec([1, S, S, 3], tf.float32))
        frozen = convert_variables_to_constants_v2(cf)
        opts = (option_builder.ProfileOptionBuilder
                .float_operation())
        opts["output"] = "none"
        info = model_analyzer.profile(frozen.graph, options=opts)
        return info.total_float_ops
    except Exception as e:
        print("   FLOPs unavailable (%s)" % type(e).__name__)
        return None

static = {}
for name, bbs in MODELS.items():
    model = build_deployable(bbs)
    p_tot = model.count_params()
    p_tr  = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    flops = count_flops(model)
    path  = "/content/%s.keras" % name
    model.save(path)
    size  = os.path.getsize(path) / 1e6
    static[name] = dict(params_M=p_tot / 1e6, trainable_M=p_tr / 1e6,
                        gflops=(flops / 2 / 1e9) if flops else None,
                        keras_MB=size)
    print("%-18s params=%7.2fM  trainable=%6.2fM  GFLOPs=%s  keras=%6.1f MB"
          % (name, p_tot / 1e6, p_tr / 1e6,
             ("%6.2f" % (flops / 2 / 1e9)) if flops else "   n/a", size))
    del model; tf.keras.backend.clear_session(); gc.collect()

# FLOPs are reported as multiply-accumulate pairs halved to GFLOPs, the usual
# convention in the vision literature. Stating.

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


MWAMNet            params=  24.81M  trainable=  3.48M  GFLOPs=  4.54  keras= 102.5 MB
DenseNet201        params=  20.83M  trainable=  2.50M  GFLOPs=  4.32  keras=  85.8 MB
MobileNetV3Large   params=   4.52M  trainable=  1.51M  GFLOPs=  0.22  keras=  18.8 MB
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
ResNet50           params=  26.23M  trainable=  2.63M  GFLOPs=  3.88  keras= 105.6 MB
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
VGG16              params=  15.77M  trainable=  1.05M  GFLOPs= 15.36  keras=  63.2 MB
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
InceptionV3        params=  24.44M  trainable=  2.63M  GFLOPs=  2.85  keras=  98.9 MB


In [ ]:
#@title 4 · GPU and CPU latency
def bench(model, device, warmup=WARMUP, timed=TIMED):
    x = tf.constant(np.random.rand(1, S, S, 3).astype("float32") * 255.0)
    with tf.device(device):
        fn = tf.function(lambda t: model(t, training=False),
                         reduce_retracing=True)
        for _ in range(warmup):
            fn(x)
        proc, rss0 = psutil.Process(), psutil.Process().memory_info().rss
        ts = []
        for _ in range(timed):
            t0 = time.perf_counter()
            _ = fn(x)
            ts.append((time.perf_counter() - t0) * 1000.0)
        rss = (proc.memory_info().rss - rss0) / 1e6
    ts = np.asarray(ts)
    return dict(mean=ts.mean(), sd=ts.std(ddof=1), median=float(np.median(ts)),
                p95=float(np.percentile(ts, 95)), rss_MB=max(rss, 0.0))

has_gpu = bool(tf.config.list_physical_devices("GPU"))
lat = {}
for name, bbs in MODELS.items():
    model = build_deployable(bbs)
    row = {}
    if has_gpu:
        try:
            tf.config.experimental.reset_memory_stats("GPU:0")
        except Exception:
            pass
        row["gpu"] = bench(model, "/GPU:0")
        try:
            row["gpu"]["peak_MB"] = tf.config.experimental.get_memory_info("GPU:0")["peak"] / 1e6
        except Exception:
            row["gpu"]["peak_MB"] = None
    row["cpu"] = bench(model, "/CPU:0", warmup=10, timed=50)
    lat[name] = row
    g = row.get("gpu")
    print("%-18s GPU %6.2f ± %4.2f ms (%5.1f FPS)   CPU %7.1f ± %5.1f ms (%4.1f FPS)"
          % (name,
             g["mean"] if g else float("nan"), g["sd"] if g else float("nan"),
             1000.0 / g["median"] if g else float("nan"),
             row["cpu"]["mean"], row["cpu"]["sd"], 1000.0 / row["cpu"]["median"]))
    del model; tf.keras.backend.clear_session(); gc.collect()

MWAMNet            GPU  30.35 ± 2.43 ms ( 33.8 FPS)   CPU   266.3 ±  76.3 ms ( 4.4 FPS)
DenseNet201        GPU  24.39 ± 3.16 ms ( 42.4 FPS)   CPU   233.8 ±  63.7 ms ( 5.0 FPS)
MobileNetV3Large   GPU   6.27 ± 0.57 ms (162.6 FPS)   CPU    36.6 ±   5.5 ms (28.5 FPS)
ResNet50           GPU   8.73 ± 2.12 ms (133.6 FPS)   CPU   121.7 ±   7.0 ms ( 8.3 FPS)
VGG16              GPU   7.82 ± 0.17 ms (127.7 FPS)   CPU   309.6 ±  83.7 ms ( 3.8 FPS)
InceptionV3        GPU  12.25 ± 2.55 ms ( 95.8 FPS)   CPU    87.1 ±   5.8 ms (11.7 FPS)


In [ ]:
#@title 5 · TFLite conversion and interpreter latency
def to_tflite(model, quantize):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                      tf.lite.OpsSet.SELECT_TF_OPS]
    if quantize:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    return conv.convert()

def tflite_latency(blob, warmup=10, timed=50):
    it = tf.lite.Interpreter(model_content=blob, num_threads=4)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    x = (np.random.rand(*inp["shape"]).astype(inp["dtype"]) * 255).astype(inp["dtype"])
    for _ in range(warmup):
        it.set_tensor(inp["index"], x); it.invoke()
    ts = []
    for _ in range(timed):
        it.set_tensor(inp["index"], x)
        t0 = time.perf_counter(); it.invoke(); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(ts)), float(np.std(ts, ddof=1))

tfl = {}
for name, bbs in MODELS.items():
    model = build_deployable(bbs)
    row = {}
    for tag, q in (("fp32", False), ("int8", True)):
        try:
            t0 = time.time()
            blob = to_tflite(model, q)
            p = "/content/%s_%s.tflite" % (name, tag)
            open(p, "wb").write(blob)
            ms, sd = tflite_latency(blob)
            row[tag] = dict(MB=len(blob) / 1e6, ms=ms, sd=sd)
            print("%-18s %-5s %6.1f MB   %7.1f ± %5.1f ms   (converted in %.0fs)"
                  % (name, tag, len(blob) / 1e6, ms, sd, time.time() - t0))
        except Exception as e:
            row[tag] = None
            print("%-18s %-5s conversion failed: %s" % (name, tag, type(e).__name__))
        gc.collect()
    tfl[name] = row
    del model; tf.keras.backend.clear_session(); gc.collect()

Saved artifact at '/tmp/tmpwjh73e7b'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134223775392848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775393616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775393424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775393808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775391312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775393040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775394576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775394384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775394768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775392656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775394000: Tens

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


MWAMNet            fp32    98.3 MB     223.7 ±  62.6 ms   (converted in 72s)
Saved artifact at '/tmp/tmp_4ylkudj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134223775392848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775393616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775393424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775393808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775391312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775393040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775394576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775394384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134223775394768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13422377539265

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


MWAMNet            int8    25.9 MB     305.8 ± 132.7 ms   (converted in 77s)
Saved artifact at '/tmp/tmp98w0o6di'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134226662620432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226662616400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226662617936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226662611792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226662617744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226662616976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226662616784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226662620624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226662614864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13422666262081

...

VGG16              int8    15.9 MB     385.2 ±  52.5 ms   (converted in 28s)
Saved artifact at '/tmp/tmpzg384igw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134226525197648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525204368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525192464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525194000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525199376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525195344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525198032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525198992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525206480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13422652519880

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


InceptionV3        fp32    97.6 MB     114.2 ±  24.6 ms   (converted in 26s)
Saved artifact at '/tmp/tmpaf1gv60d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134226525197648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525204368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525192464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525194000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525199376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525195344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525198032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525198992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134226525206480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13422652519880

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


InceptionV3        int8    24.7 MB     107.7 ±  19.7 ms   (converted in 27s)


In [ ]:
#@title 6 · Assemble, save and export LaTeX
rows = []
for name in MODELS:
    s, l, t = static[name], lat[name], tfl[name]
    g = l.get("gpu")
    rows.append({
        "Model": name,
        "Params (M)": round(s["params_M"], 2),
        "Trainable (M)": round(s["trainable_M"], 2),
        "GFLOPs": round(s["gflops"], 2) if s["gflops"] else None,
        "Keras (MB)": round(s["keras_MB"], 1),
        "TFLite fp32 (MB)": round(t["fp32"]["MB"], 1) if t["fp32"] else None,
        "TFLite int8 (MB)": round(t["int8"]["MB"], 1) if t["int8"] else None,
        "GPU (ms)": round(g["mean"], 2) if g else None,
        "GPU SD": round(g["sd"], 2) if g else None,
        "GPU FPS": round(1000.0 / g["median"], 1) if g else None,
        "CPU (ms)": round(l["cpu"]["mean"], 1),
        "CPU FPS": round(1000.0 / l["cpu"]["median"], 2),
        "TFLite int8 (ms)": round(t["int8"]["ms"], 1) if t["int8"] else None,
        "GPU peak (MB)": round(g["peak_MB"], 1) if g and g.get("peak_MB") else None,
    })

E = pd.DataFrame(rows).sort_values("Params (M)")
E.to_csv(os.path.join(OUT, "results", "efficiency.csv"), index=False)
if MIRROR:
    E.to_csv(os.path.join(MIRROR, "efficiency.csv"), index=False)
display(E)

sub = E[["Model", "Params (M)", "GFLOPs", "TFLite int8 (MB)",
         "GPU (ms)", "GPU FPS", "CPU (ms)", "TFLite int8 (ms)"]]
print("\nLaTeX:\n")
print(sub.to_latex(index=False, escape=False, na_rep="n/a", float_format="%.2f",
      caption=("Computational cost of each deployable pipeline (preprocessing, "
               "backbone(s) and classifier head), batch size 1. Latency is the mean "
               "of 200 timed inferences after 50 warm-up runs on an NVIDIA Tesla T4; "
               "CPU and TFLite figures are measured on the same host. FLOPs are "
               "reported as multiply-accumulate pairs."),
      label="tab:efficiency"))

mw = E[E.Model == "MWAMNet"].iloc[0]
lightest = E.iloc[0]
print("\nFor the discussion: MWAMNet is %.1fx the parameters and %.1fx the GPU latency "
      "of %s." % (mw["Params (M)"] / lightest["Params (M)"],
                  mw["GPU (ms)"] / lightest["GPU (ms)"], lightest["Model"]))

,Model,Params (M),Trainable (M),GFLOPs,Keras (MB),TFLite fp32 (MB),TFLite int8 (MB),GPU (ms),GPU SD,GPU FPS,CPU (ms),CPU FPS,TFLite int8 (ms),GPU peak (MB)
2,MobileNetV3Large,4.52,1.51,0.22,18.8,17.9,4.8,6.27,0.57,162.6,36.6,28.50,45.1,31.0
4,VGG16,15.77,1.05,15.36,63.2,63.1,15.9,7.82,0.17,127.7,309.6,3.75,385.2,2414.6
1,DenseNet201,20.83,2.50,4.32,85.8,82.5,21.6,24.39,3.16,42.4,233.8,5.00,215.9,116.6
5,InceptionV3,24.44,2.63,2.85,98.9,97.6,24.7,12.25,2.55,95.8,87.1,11.68,107.7,916.0
0,MWAMNet,24.81,3.48,4.54,102.5,98.3,25.9,30.35,2.43,33.8,266.3,4.42,305.8,414.9
3,ResNet50,26.23,2.63,3.88,105.6,104.4,26.6,8.73,2.12,133.6,121.7,8.32,141.2,1312.0



LaTeX:

\begin{table}
\caption{Computational cost of each deployable pipeline (preprocessing, backbone(s) and classifier head), batch size 1. Latency is the mean of 200 timed inferences after 50 warm-up runs on an NVIDIA Tesla T4; CPU and TFLite figures are measured on the same host. FLOPs are reported as multiply-accumulate pairs.}
\label{tab:efficiency}
\begin{tabular}{lrrrrrrr}
\toprule
Model & Params (M) & GFLOPs & TFLite int8 (MB) & GPU (ms) & GPU FPS & CPU (ms) & TFLite int8 (ms) \\
\midrule
MobileNetV3Large & 4.52 & 0.22 & 4.80 & 6.27 & 162.60 & 36.60 & 45.10 \\
VGG16 & 15.77 & 15.36 & 15.90 & 7.82 & 127.70 & 309.60 & 385.20 \\
DenseNet201 & 20.83 & 4.32 & 21.60 & 24.39 & 42.40 & 233.80 & 215.90 \\
InceptionV3 & 24.44 & 2.85 & 24.70 & 12.25 & 95.80 & 87.10 & 107.70 \\
MWAMNet & 24.81 & 4.54 & 25.90 & 30.35 & 33.80 & 266.30 & 305.80 \\
ResNet50 & 26.23 & 3.88 & 26.60 & 8.73 & 133.60 & 121.70 & 141.20 \\
\bottomrule
\end{tabular}
\end{table}


For the discussion: MWAMNet is 5.5x 

In [ ]:
#@title 7 · Files produced
for f in sorted(os.listdir("/content")):
    if f.endswith((".tflite", ".keras")):
        print("  /content/%-34s %8.1f MB" % (f, os.path.getsize("/content/" + f) / 1e6))
print("\n  results/efficiency.csv")
print("\nSend me efficiency.csv and the LaTeX block from cell 6.")

  /content/DenseNet201.keras                      85.8 MB
  /content/DenseNet201_fp32.tflite                82.5 MB
  /content/DenseNet201_int8.tflite                21.6 MB
  /content/InceptionV3.keras                      98.9 MB
  /content/InceptionV3_fp32.tflite                97.6 MB
  /content/InceptionV3_int8.tflite                24.7 MB
  /content/MWAMNet.keras                         102.5 MB
  /content/MWAMNet_fp32.tflite                    98.3 MB
  /content/MWAMNet_int8.tflite                    25.9 MB
  /content/MobileNetV3Large.keras                 18.8 MB
  /content/MobileNetV3Large_fp32.tflite           17.9 MB
  /content/MobileNetV3Large_int8.tflite            4.8 MB
  /content/ResNet50.keras                        105.6 MB
  /content/ResNet50_fp32.tflite                  104.4 MB
  /content/ResNet50_int8.tflite                   26.6 MB
  /content/VGG16.keras                            63.2 MB
  /content/VGG16_fp32.tflite                      63.1 MB
  /content/VGG